# Notebook 02 — Modeling & Class-Imbalance Handling
**Master Playbook Section 5, 6, 10**

Depends on Notebook 01 having verified `creditcard.csv` loads cleanly. This
notebook runs the real Stage A → Stage B → champion-selection → cost-optimal
threshold pipeline, using the already-verified `class_imbalance_utils.py` and
`model_benchmark.py` modules — nothing here re-implements that logic ad hoc.

In [ ]:
import sys, os, pickle
sys.path.insert(0, os.path.abspath("../"))
import numpy as np
import pandas as pd

import class_imbalance_utils as ciu
import model_benchmark as mb

from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
DATA_PATH = "creditcard.csv"

df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c not in ("Time", "Class")]
X = df[feature_cols]
y = df["Class"]
amounts = df["Amount"].values
print(f"X shape: {X.shape}, fraud rate: {y.mean():.6%}")

## Class-imbalance technique comparison (Section 6)
Compares class weighting vs. threshold-moving vs. SMOTE — SMOTE applied only inside each CV fold's training split, never before, to avoid leakage.

In [ ]:
def build_model_fn(class_weight=None):
    # class_weight is a dict (Strategy 1: class weighting) or None (Strategies 2/3),
    # per class_imbalance_utils.compare_imbalance_strategies's documented contract.
    return RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1, class_weight=class_weight)

imbalance_results = ciu.compare_imbalance_strategies(X, y, build_model_fn, n_splits=5)
for r in imbalance_results:
    print(r)
winner = ciu.winning_strategy(imbalance_results)
print("\nWinning strategy (by real mean CV PR-AUC):", winner)

## Stage A — screen 4 candidates on one split (Section 6)

In [ ]:
candidates = {"RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)}

try:
    from xgboost import XGBClassifier
    candidates["XGBoost"] = XGBClassifier(random_state=RANDOM_SEED, eval_metric="aucpr", n_jobs=-1)
except ImportError:
    print("xgboost not installed — skipping (pip install xgboost)")

try:
    from lightgbm import LGBMClassifier
    candidates["LightGBM"] = LGBMClassifier(random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1)
except ImportError:
    print("lightgbm not installed — skipping (pip install lightgbm)")

try:
    from catboost import CatBoostClassifier
    candidates["CatBoost"] = CatBoostClassifier(random_state=RANDOM_SEED, verbose=False)
except ImportError:
    print("catboost not installed — skipping (pip install catboost)")

stage_a = mb.run_stage_a_screening(X, y, candidates)
for name, result in sorted(stage_a.items(), key=lambda kv: -kv[1].pr_auc):
    print(name, result)

## Stage B — real 5-fold CV on the top 2, with bootstrap CI (Section 6)

In [ ]:
top2_names = sorted(stage_a, key=lambda k: -stage_a[k].pr_auc)[:2]
top2 = {name: candidates[name] for name in top2_names}
print("Top 2 advancing to Stage B:", top2_names)

stage_b = mb.run_stage_b_cv(X, y, top2, n_splits=5, n_bootstrap=1000)
for name, result in stage_b.items():
    print(name, result)

champion_name = mb.select_champion(stage_b)
champion_model = top2[champion_name]
print("\nCHAMPION (real, by mean CV PR-AUC):", champion_name)

## Temporal-split validation (Section 6, 19.4)
Checks whether the champion's performance holds on a time-ordered split, not just random k-fold.

In [ ]:
temporal_pr_auc = mb.temporal_split_validation(df, "Time", feature_cols, "Class", champion_model)
print(f"Temporal-split PR-AUC: {temporal_pr_auc:.4f}  (compare honestly to Stage B's CV PR-AUC above)")

## Fit champion on full data, get out-of-fold-style scores for thresholding

In [ ]:
from sklearn.model_selection import cross_val_predict

champion_model.fit(X, y)
y_scores = cross_val_predict(champion_model, X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]

## Cost-optimal threshold selection (Section 10)
Uses the two real, sourced cost benchmarks — not an arbitrary 0.5 cutoff.

In [ ]:
threshold_result = ciu.best_threshold_by_cost(
    y_true=y.values,
    y_scores=y_scores,
    fn_cost_per_dollar_lost=4.41,      # LexisNexis True Cost of Fraud (TM)
    fp_cost_multiplier_vs_fraud=9.2,   # Aite-Novarica / Statista via Riskified
    amounts=amounts,
)
print(threshold_result)
CHOSEN_THRESHOLD = threshold_result["threshold"]

## External benchmark comparison (Section 19.4)

In [ ]:
from sklearn.metrics import precision_score, recall_score

y_pred = (y_scores >= CHOSEN_THRESHOLD).astype(int)
real_precision = precision_score(y, y_pred)
real_recall = recall_score(y, y_pred)
print(f"Real precision: {real_precision:.4f}, real recall: {real_recall:.4f}")

benchmark_check = mb.compare_to_external_benchmark(real_precision, real_recall)
print(benchmark_check)

## Save the champion model for Notebooks 03 and 04, and for `deployment/`

In [ ]:
with open("champion_model.pkl", "wb") as f:
    pickle.dump(champion_model, f)
print("Saved champion_model.pkl — copy this into deployment/model/ for Phase 5.")

## Your real conclusions (fill in AFTER running)

- **Winning class-imbalance strategy:** _[from the comparison above]_
- **Champion model:** _[name]_, CV PR-AUC _[value]_ (bootstrap 95% CI: _[lo–hi]_)
- **Temporal-split PR-AUC:** _[value]_ — consistent with CV? _[yes/no + why]_
- **Chosen threshold and its real cost basis:** _[value]_
- **vs. external benchmark:** _[consistent / investigate — and why]_
